In [3]:
# Load packages 
import datasets 
datasets.disable_caching()
from transformers import set_seed 
set_seed(42)

In [4]:
#!pip install datasets

In [5]:
# KTH has 7 core + 12 electives = 19 courses 
# NTU has 13 courses (plus the capstone project) 
# CMU has 20 courses 

import numpy as np 
np.random.seed(seed=42) 
KTH = np.random.randint(19, size=3)
NTU = np.random.randint(13, size=3) 
CMU = np.random.randint(19, size=3) 
print(KTH)
print(NTU)
print(CMU)
# corresponds to: 
# Ethical Hacking (EN2720) 
# Language-based Security (DD2525)
# Building Networked Systems Security (EP2520) 
# Network Security 
# Security Monitoring and Threat Detection (SE6014) 
# Privacy Preserving Technologies and Security in AI 
# Advanced Real-World Data Networks (14-760) 
# Security in Networked Systems (14-742) 
# Information Security Policy and Management (14-788) 

[ 6 14 10]
[ 7 12  4]
[ 6 18 10]


In [7]:
import pandas as pd 
df = pd.read_excel('/nobackup/proj/disk/naiss2024-22-903/personal/LLMedu/data/courses_file.ods', engine='odf')
print(df)

  University                                        Course name   \
0        KTH                                    Ethical Hacking    
1        KTH                            Language-Based Security    
2        KTH                Building Networked Systems Security    
3        NTU                                   Network Security    
4        NTU                                  Software Security    
5        NTU   Privacy Preserving Technologies & Security in AI    
6        CMU                       Advanced Real-World Networks    
7        CMU                      Security in Networked Systems    
8        CMU         Information Security Policy and Management    

  Course code                                  Course Description   
0       EN2720   The main activity of the course is a project w...  
1       DD2525   Introduction to language-based security. Funda...  
2       EP2520   The course trains students to handle contempor...  
3       SE6011   Introduction to networking

In [9]:
df.keys()
course_names = df.get(df.keys()[1])
print(course_names)
course_descriptions = df.get(df.keys()[-1])

0                                     Ethical Hacking 
1                             Language-Based Security 
2                 Building Networked Systems Security 
3                                    Network Security 
4                                   Software Security 
5    Privacy Preserving Technologies & Security in AI 
6                        Advanced Real-World Networks 
7                       Security in Networked Systems 
8          Information Security Policy and Management 
Name: Course name , dtype: str


In [10]:
# Load Qwen2-7B-Instruct model. 
import sys
sys.path.append("/nobackup/proj/disk/naiss2024-22-903/personal/LLMedu")
from utils.llm_utils import submit_message_LLM, topic_prompt 
# gets saved at naiss2024-22-903/.cache/huggingface/models
from transformers import AutoModelForCausalLM, AutoTokenizer
API_KEY="<YOUR-APKI-KEY-HERE>"
#model_path = "deepseek-ai/deepseek-r1-distill-qwen-14B" 
# change later to: model_id = "deepseek-ai/DeepSeek-R1
model_path="Qwen/Qwen2.5-7B-Instruct"
device = "cuda" # the device to load the model onto

model2 = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer2 = AutoTokenizer.from_pretrained(model_path)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [11]:
# Load trained DistilBERT model 
from sklearn.metrics import multilabel_confusion_matrix
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import DataCollatorWithPadding
# load model from a working checkpoint 
model_name='/nobackup/proj/disk/naiss2024-22-903/personal/LLMedu/DistilBERTv1final/checkpoint-430' #'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)             # the tokenizer is the same for all folds 
data_collator = DataCollatorWithPadding(tokenizer=tokenizer) # the data collator is the same for all folds 
# misc: (this includes Computer Science, Business and Law, Communication and Networking, Information Technology, Cyberspace Practice, Pedagogy, and Intelligence)
KAs = {"0": "miscellaneous",
           "1": "data security", 
           "2": "software security",
           "3": "component security", 
           "4": "connection security", 
           "5": "system security", 
           "6": "human security", 
           "7": "organizational security",
           "8": "societal security"}

def sigmoid(x):
   return 1/(1 + np.exp(-x))
    
model = AutoModelForSequenceClassification.from_pretrained(model_name, output_hidden_states=True,
                                                                        problem_type="multi_label_classification",
                                                                        num_labels=9, dropout=0.2)
model.eval()

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.2, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.2, inplace=False)


In [12]:
# loop over all courses: 
import re
from datasets import Dataset 
from utils.load_data import clean_text
device = 'cuda:0'
KA_labels = [] 
for idx in range(0,9): 
    print(idx)
    description = course_descriptions[idx]                             # read course descriptions
    topic = course_names[idx]                                          # read course topics 
    print('name: ',topic)
    #print('description: ', description)
    #lo = LO[idx]                                                       # read learning objectives 
    message = topic_prompt(topic, description)                         # create prompt based on topic + description
    response = submit_message_LLM(model2, message, tokenizer2, device)  # put the formatted prompt through the LLM 

    #print('response: ', response)
    subtopic = response.split('- ')[1:] # isolates subtopics 
    subtopics = {}
    for i, sub in enumerate(subtopic): 
        # if key does not exist, create it, otherwise append? 
        #print(clean_text(sub))
        print(sub)
        subtopics[str(i)] = [clean_text(sub)]

    # make them into the right format (a dataset?) 
    course = Dataset.from_dict(subtopics)
    llm_labels = [] 
    tokenized_course = {}
    for i in range(len(course.features)): 
        #print(course[str(i)])
        tokenized_course[str(i)] = tokenizer(course[str(i)], return_tensors='pt') #, padding='max_length', max_length=64)
        with torch.no_grad(): 
            predictions= model(**tokenized_course[str(i)])
            predictions = sigmoid(predictions.logits).detach().cpu().numpy()
            predictions = (predictions > 0.5).astype(int).reshape(-1)
            llm_labels.append(predictions)
            print([x for x in predictions if x >0])
    # combine all labels of the course. 
    llm_labels2 = np.array(llm_labels)
    
    final_output = np.sum(llm_labels2,axis=0) / (np.sum(llm_labels2)) #/ len(course.features) # divide by the total number of 'items' 

    # now compare this to feeding just the course name 
    with torch.no_grad(): 
        predictions = model(**tokenizer(clean_text(topic), return_tensors='pt'))
        predictions = sigmoid(predictions.logits.detach().cpu().numpy())
        predictions = (predictions > 0.5).astype(int).reshape(-1)
        #print('course name: ',predictions)
        print([x for x in predictions if x >0])
        llm_labels.append(predictions)
    print(predictions)
    llm_labels = np.array(llm_labels)
    final_output2 = np.sum(llm_labels,axis=0) / (np.sum(llm_labels)) #/ len(course.features) # divide by the total number of 'items' 
    print('LLM Labels: ',final_output2)
    KA_labels.append(final_output2) 
    print([KAs[str(j.item())] for j in np.where(final_output2>0)[0]])
    

0
name:  Ethical Hacking 
Project-based learning

Virtual environment setup

Network scanning techniques

Vulnerability assessment

Exploit development

Command and control mechanisms

Password cracking methods
[1]
[1]
[1]
[1, 1]
[]
[1]
[1]
[1]
[0 0 0 0 0 0 0 0 1]
LLM Labels:  [0.25  0.    0.125 0.125 0.25  0.    0.125 0.    0.125]
['miscellaneous', 'software security', 'component security', 'connection security', 'human security', 'societal security']
1
name:  Language-Based Security 
Fundamental principles, models and concepts for computer security

Software security by information flow control

Web application and database security

Security for mobile applications

Hot topics in computer security

State-of-the-art in programming language for security
[]
[1]
[1, 1]
[1, 1]
[1]
[1]
[]
[0 0 0 0 0 0 0 0 0]
LLM Labels:  [0.         0.         0.14285714 0.         0.         0.
 0.28571429 0.57142857 0.        ]
['software security', 'human security', 'organizational security']
2
name:  

### All courses 

In [2]:
# Take in a course 
val_dataset = datasets.load_dataset("csv",data_files={"train": "data/train_data.csv"}, split='train')

In [3]:
import pandas as pd 
df = pd.read_excel('data/courses.ods', engine='odf')
print(df)

   University                     Type   Credits   \
0          LU                 Elective        7.5   
1          LU                 Elective        7.5   
2          LU                 Elective        7.5   
3          LU                 Elective        7.5   
4         KTH                Mandatory        4.5   
5         KTH                Mandatory        3.0   
6         KTH                Mandatory        2.0   
7         KTH                Mandatory        7.5   
8         KTH                Mandatory        7.5   
9         KTH                Mandatory        7.5   
10        KTH                Mandatory        7.5   
11        KTH   Conditionally Elective        7.5   
12        KTH   Conditionally Elective        7.5   
13        KTH   Conditionally Elective        7.5   
14        KTH   Conditionally Elective        7.5   
15        KTH   Conditionally Elective        7.5   
16        KTH   Conditionally Elective        7.5   
17        KTH   Conditionally Elective        

In [25]:
course_names = df.get(df.keys()[3])
print(course_names)
course_descriptions = df.get(df.keys()[-1])
LO = df.get(df.keys()[5])
#print(LO)

0                                Advanced Web Security 
1                           Secure Systems Engineering 
2                                         Cryptography 
3                                         Web Security 
4     Theory and Methodology of Science (Natural and...
5     Theory of Science and Scientific methods in Cy...
6        The Cybersecurity Engineer's Role in Society  
7                              Cybersecurity Overview  
8          Cybersecurity in a Socio-Technical Context  
9                                Applied Cryptography  
10                                    Ethical Hacking  
11                         Foundations of Cryptography 
12                     Privacy Enhancing Technologies  
13                  Project course in System Security  
14                            Language-Based Security  
15    Cyber-Physical Security in Time-Critical Syste...
16                         Networked Systems Security  
17                Advanced Networked Systems Sec

In [6]:
idx = 11 # 11:23 
print('Description: ',course_descriptions[idx])
description = course_descriptions[idx]
print('Course name: ',course_names[idx]) 
topic = course_names[idx]

Description:  Classic cryptosystems. What does secure encryption mean? Background in information theory, entropy. Symmetric encryption algorithms such as Advanced Encryption Standard (AES). Open key systems for encryption and digital signatures e.g. RSA, ElGamal and Schnorr signatures. Cryptographically secure hash functions in theory and practice (SHA). Properties and examples of pseudo-random number generators. Connections to complexity theory. 
Course name:  Foundations of Cryptography 


In [7]:
# Load Qwen2-7B-Instruct model. 
from utils.llm_utils import submit_message_LLM, topic_prompt 
# gets saved at naiss2024-22-903/.cache/huggingface/models
from transformers import AutoModelForCausalLM, AutoTokenizer
API_KEY="<YOUR-APKI-KEY-HERE>"
#model_path = "deepseek-ai/deepseek-r1-distill-qwen-14B" 
# change later to: model_id = "deepseek-ai/DeepSeek-R1
model_path="Qwen/Qwen2.5-7B-Instruct"
device = "cuda" # the device to load the model onto

model2 = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer2 = AutoTokenizer.from_pretrained(model_path)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [39]:
# Load trained DistilBERT model 
from sklearn.metrics import multilabel_confusion_matrix
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import DataCollatorWithPadding
import sys
import torch
import numpy as np
sys.path.append("/mimer/NOBACKUP/groups/naiss2024-22-903/LLMedu")

# load model from a working checkpoint 
model_name='DistilBERTv1final/checkpoint-430' #'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)             # the tokenizer is the same for all folds 
data_collator = DataCollatorWithPadding(tokenizer=tokenizer) # the data collator is the same for all folds 
# misc: (this includes Computer Science, Business and Law, Communication and Networking, Information Technology, Cyberspace Practice, Pedagogy, and Intelligence)
KAs = {"0": "miscellaneous",
           "1": "data security", 
           "2": "software security",
           "3": "component security", 
           "4": "connection security", 
           "5": "system security", 
           "6": "human security", 
           "7": "organizational security",
           "8": "societal security"}

def sigmoid(x):
   return 1/(1 + np.exp(-x))
    
model = AutoModelForSequenceClassification.from_pretrained(model_name, output_hidden_states=True,
                                                                        problem_type="multi_label_classification",
                                                                        num_labels=9, dropout=0.2)
model.eval()

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.2, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.2, inplace=False)


In [42]:
# loop over all courses: 
import re
from datasets import Dataset 
from utils.load_data import clean_text
device = 'cuda:0'
KA_labels = [] 
for idx in range(0,24): 
    print(idx)
    description = course_descriptions[idx]                             # read course descriptions
    topic = course_names[idx]                                          # read course topics 
    print('name: ',topic)
    #print('description: ', description)
    lo = LO[idx]                                                       # read learning objectives 
    message = topic_prompt(topic, description)                         # create prompt based on topic + description
    response = submit_message_LLM(model2, message, tokenizer2, device)  # put the formatted prompt through the LLM 

    #print('response: ', response)
    subtopic = response.split('- ')[1:] # isolates subtopics 
    subtopics = {}
    for i, sub in enumerate(subtopic): 
        # if key does not exist, create it, otherwise append? 
        #print(clean_text(sub))
        subtopics[str(i)] = [clean_text(sub)]

    # make them into the right format (a dataset?) 
    course = Dataset.from_dict(subtopics)
    llm_labels = [] 
    tokenized_course = {}
    for i in range(len(course.features)): 
        #print(course[str(i)])
        tokenized_course[str(i)] = tokenizer(course[str(i)], return_tensors='pt') #, padding='max_length', max_length=64)
        with torch.no_grad(): 
            predictions= model(**tokenized_course[str(i)])
            predictions = sigmoid(predictions.logits).detach().cpu().numpy()
            predictions = (predictions > 0.5).astype(int).reshape(-1)
            llm_labels.append(predictions)
    # combine all labels of the course. 
    llm_labels2 = np.array(llm_labels)
    #print(llm_labels2)
    final_output = np.sum(llm_labels2,axis=0) / (np.sum(llm_labels2)) #/ len(course.features) # divide by the total number of 'items' 
    #print('first method: ',final_output)
    #print([KAs[str(j.item())] for j in np.where(final_output>0)[0]])

    # do the same with the LOs 
    message = topic_prompt(topic, lo)                         # create prompt based on topic + description
    response = submit_message_LLM(model2, message, tokenizer2, device)  # put the formatted prompt through the LLM 

    #print('response: ', response)
    subtopic = response.split('- ')[1:] # isolates subtopics 
    subtopics = {}
    for i, sub in enumerate(subtopic): 
        subtopics[str(i)] = [clean_text(sub)]

    # make them into the right format (a dataset?) 
    course = Dataset.from_dict(subtopics)
    tokenized_course = {}
    for i in range(len(course.features)): 
        tokenized_course[str(i)] = tokenizer(course[str(i)], return_tensors='pt') #, padding='max_length', max_length=64)
        with torch.no_grad(): 
            predictions= model(**tokenized_course[str(i)])
            predictions = sigmoid(predictions.logits).detach().cpu().numpy()
            predictions = (predictions > 0.5).astype(int).reshape(-1)
            llm_labels.append(predictions)
    # combine all labels of the course. 
    

    # now compare this to feeding just the course name 
    with torch.no_grad(): 
        predictions = model(**tokenizer(clean_text(topic), return_tensors='pt'))
        predictions = sigmoid(predictions.logits.detach().cpu().numpy())
        predictions = (predictions > 0.5).astype(int).reshape(-1)
        #print('course name: ',predictions)
        llm_labels.append(predictions)
        # and the learning objectives 
        #predictions = model(**tokenizer(clean_text(lo), return_tensors='pt'))
        #predictions = sigmoid(predictions.logits.detach().cpu().numpy())
        #predictions = (predictions > 0.5).astype(int).reshape(-1)
        #print('learning obj: ',predictions)
        # and the entire description at once 
        #predictions = model(**tokenizer(clean_text(response), return_tensors='pt'))
        #predictions = sigmoid(predictions.logits.detach().cpu().numpy())
        #predictions = (predictions > 0.5).astype(int).reshape(-1)
        #print('course descr: ',predictions)
    llm_labels = np.array(llm_labels)
    final_output2 = np.sum(llm_labels,axis=0) / (np.sum(llm_labels)) #/ len(course.features) # divide by the total number of 'items' 
    print('first method LOs: ',final_output2)
    KA_labels.append(final_output2) 
    print([KAs[str(j.item())] for j in np.where(final_output2>0)[0]])
    

0
name:  Advanced Web Security 
first method LOs:  [0.         0.38461538 0.07692308 0.         0.30769231 0.
 0.         0.15384615 0.07692308]
['data security', 'software security', 'connection security', 'organizational security', 'societal security']
1
name:  Secure Systems Engineering 
first method LOs:  [0.07142857 0.         0.21428571 0.         0.14285714 0.
 0.         0.57142857 0.        ]
['miscellaneous', 'software security', 'connection security', 'organizational security']
2
name:  Cryptography 
first method LOs:  [0.02083333 0.83333333 0.04166667 0.         0.         0.02083333
 0.02083333 0.0625     0.        ]
['miscellaneous', 'data security', 'software security', 'system security', 'human security', 'organizational security']
3
name:  Web Security 
first method LOs:  [0.02857143 0.34285714 0.02857143 0.         0.08571429 0.02857143
 0.11428571 0.37142857 0.        ]
['miscellaneous', 'data security', 'software security', 'connection security', 'system security', 

/local/tmp.4542987/ipykernel_196890/2576364546.py:40: RuntimeWarning: invalid value encountered in double_scalars
  final_output = np.sum(llm_labels2,axis=0) / (np.sum(llm_labels2)) #/ len(course.features) # divide by the total number of 'items'


first method LOs:  [0.         0.         0.33333333 0.         0.33333333 0.
 0.         0.16666667 0.16666667]
['software security', 'connection security', 'organizational security', 'societal security']


In [51]:
#print(KA_labels)
lab = np.array(KA_labels)
print(np.sum(lab[:11,:],axis=0))

[0.76726617 2.51739307 0.48382451 0.         0.89421828 0.20325092
 0.50153627 3.89198105 1.74052973]


In [53]:
sum_labels = np.sum(lab[:11,:],axis=0) / np.sum(lab[:11,:])
print(sum_labels)

[0.06975147 0.22885392 0.04398405 0.         0.08129257 0.01847736
 0.04559421 0.35381646 0.15822998]


### One by one 

In [15]:
from utils.load_data import clean_text
message = topic_prompt(topic, description)      # create prompt based on topic + description
device = 'cuda:0'
response = submit_message_LLM(model2, message, tokenizer, device)  # put the formatted prompt through the LLM 
#print('response: ',response)
subtopics = response.split("\n")                # split response into subtopics
for sub in subtopics: 
    #sub = clean_text(sub[2:])
    print(sub)
    #topics.append(sub)                      # add subtopics and the original label to lists 
    #labels.append(label)
    #descriptions.append(description)

ValueError: Cannot use chat template functions because tokenizer.chat_template is not set and no template argument was passed! For information about writing templates and setting the tokenizer.chat_template attribute, please see the documentation at https://huggingface.co/docs/transformers/main/en/chat_templating

In [37]:
import re
output = response.split('1.')[1] # removes all 'thinking' output 
#print(output)
subtopic = re.split(r'\s\d+\.\s', output)
subtopics = {}
for i, sub in enumerate(subtopic): 
    # if key does not exist, create it, otherwise append? 
    #print(clean_text(sub))
    subtopics[str(i)] = [clean_text(sub)]
print(subtopics)

{'0': [' internet and tcp ip networks,'], '1': ['mobile voice and data networks,'], '2': ['wireless local and personal networks,'], '3': ['wireless sensor networks,'], '4': ['mobile ad hoc and hybrid networks (including vanets),'], '5': ['security concepts and technologies,'], '6': ['joint security requirements,'], '7': ['functions determining security solutions,'], '8': ['design decisions']}


In [38]:
# make them into the right format (a dataset?) 
from datasets import Dataset 
course = Dataset.from_dict(subtopics)
print(course)

Dataset({
    features: ['0', '1', '2', '3', '4', '5', '6', '7', '8'],
    num_rows: 1
})


In [46]:
# Load trained DistilBERT model 
from sklearn.metrics import multilabel_confusion_matrix
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import DataCollatorWithPadding
import sys
sys.path.append("/mimer/NOBACKUP/groups/naiss2024-22-903/LLMedu")

# load model from a working checkpoint 
model_name='DistilBERTv1final/checkpoint-430' #'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)             # the tokenizer is the same for all folds 
data_collator = DataCollatorWithPadding(tokenizer=tokenizer) # the data collator is the same for all folds 


In [99]:
import torch
import numpy as np # misc: (this includes Computer Science, Business and Law, Communication and Networking, Information Technology, Cyberspace Practice, Pedagogy, and Intelligence)
KAs = {"0": "miscellaneous",
           "1": "data security", 
           "2": "software security",
           "3": "component security", 
           "4": "connection security", 
           "5": "system security", 
           "6": "human security", 
           "7": "organizational security",
           "8": "societal security"}

def sigmoid(x):
   return 1/(1 + np.exp(-x))
    
model = AutoModelForSequenceClassification.from_pretrained(model_name, output_hidden_states=True,
                                                                        problem_type="multi_label_classification",
                                                                        num_labels=9, dropout=0.2)
model.eval()
llm_labels = [] 
for i in range(len(course.features)): 
    print(course[str(i)])
    tokenized_course[str(i)] = tokenizer(course[str(i)], return_tensors='pt') #, padding='max_length', max_length=64)
    with torch.no_grad(): 
        predictions= model(**tokenized_course[str(i)])
        predictions = sigmoid(predictions.logits).detach().cpu().numpy()
        predictions = (predictions > 0.5).astype(int).reshape(-1)
        #if sum(llm_label) == 0: # if we also want to use inputs that are not confident enough 
        #3    llm_label = np.zeros((1,9))
        #    idx = np.argmax(predictions)
        #    llm_label[:,idx] = 1 # adjust if there are multiple courses passed at the same time. 
        llm_labels.append(predictions)
# combine all labels of the course. 
llm_labels = np.array(llm_labels)
print(llm_labels)
final_output = np.sum(llm_labels,axis=0) / (np.sum(llm_labels)) #/ len(course.features) # divide by the total number of 'items' 
print(final_output)
print([KAs[str(j.item())] for j in np.where(final_output>0)[0]])
# also add the course title itself 

[' internet and tcp ip networks,']
['mobile voice and data networks,']
['wireless local and personal networks,']
['wireless sensor networks,']
['mobile ad hoc and hybrid networks (including vanets),']
['security concepts and technologies,']
['joint security requirements,']
['functions determining security solutions,']
['design decisions']
[[0 0 0 0 1 0 0 0 0]
 [0 0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0 0]
 [0 0 0 0 1 0 0 0 0]
 [0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 1]]
[0.  0.  0.  0.  0.6 0.  0.  0.2 0.2]
['connection security', 'organizational security', 'societal security']


### Old code 

In [ ]:
if sum(llm_label) == 0: 
            llm_label = np.zeros((1,9))
            idx = np.argmax(predictions)
            llm_label[:,idx] = 1 # adjust if there are multiple courses passed at the same time. 
        #print(llm_label)
        llm_labels.append(llm_label)
        labels = [KAs[str(j.item())] for j in np.where(llm_label>0)[0]]
        print(labels)
# combine all labels of the course. 
print(llm_labels)

In [79]:
import torch
import numpy as np # misc: (this includes Computer Science, Business and Law, Communication and Networking, Information Technology, Cyberspace Practice, Pedagogy, and Intelligence)
KAs = {"0": "miscellaneous",
           "1": "data security", 
           "2": "software security",
           "3": "component security", 
           "4": "connection security", 
           "5": "system security", 
           "6": "human security", 
           "7": "organizational security",
           "8": "societal security"}

def sigmoid(x):
   return 1/(1 + np.exp(-x))
    
model = AutoModelForSequenceClassification.from_pretrained(model_name, output_hidden_states=True,
                                                                        problem_type="multi_label_classification",
                                                                        num_labels=9, dropout=0.2)
model.eval()
llm_labels = [] 
for i in range(len(course.features)): 
    print(course[str(i)])
    tokenized_course[str(i)] = tokenizer(course[str(i)], return_tensors='pt') #, padding='max_length', max_length=64)
    with torch.no_grad(): 
        predictions= model(**tokenized_course[str(i)])
       # print(predictions)
        predictions = sigmoid(predictions.logits).detach().cpu().numpy()
        #print(predictions)
        llm_label = (predictions > 0.5).astype(int).reshape(-1)
        if sum(llm_label) == 0: 
            llm_label = np.zeros((1,9))
            idx = np.argmax(predictions)
            llm_label[:,idx] = 1 # adjust if there are multiple courses passed at the same time. 
        #print(llm_label)
        llm_labels.append(llm_label)
        labels = [KAs[str(j.item())] for j in np.where(llm_label>0)[0]]
        print(labels)
# combine all labels of the course. 
print(llm_labels)


[' internet and tcp ip networks,']
[[0.0284835  0.02497495 0.00933375 0.01276194 0.90398973 0.02706327
  0.0147735  0.05587436 0.01742662]]
['connection security']
['mobile voice and data networks,']
[[0.03850658 0.29299477 0.00253389 0.00262391 0.17751783 0.04129511
  0.25054374 0.13214242 0.0284964 ]]
['miscellaneous']
['wireless local and personal networks,']
[[0.01354401 0.12714781 0.00312637 0.00329171 0.51857793 0.01852061
  0.13320011 0.16161613 0.0428065 ]]
['connection security']
['wireless sensor networks,']
[[0.02087427 0.10677362 0.00225869 0.00296906 0.62087655 0.03002157
  0.05837579 0.12195189 0.0210558 ]]
['connection security']
['mobile ad hoc and hybrid networks (including vanets),']
[[0.09991411 0.02833556 0.00209423 0.00191664 0.18079734 0.01336825
  0.05636595 0.27459657 0.02014716]]
['miscellaneous']
['security concepts and technologies,']
[[0.11782232 0.02469447 0.01037377 0.00820424 0.01068093 0.26913506
  0.00614757 0.28396702 0.0035062 ]]
['miscellaneous']
['j

In [ ]:
# Then save the course and pass all the descriptions through the trained network 

In [ ]:
# save the outputs 